<a href="https://colab.research.google.com/github/VARUN-OFFICIAL-24/rag-based-qa/blob/main/notebooks/rag_multiple_documents.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
file_path = r"C:\Users\darkw\PycharmProjects\LetsBegin\HTML"

In [ ]:
pip install langchain_chroma

Note: you may need to restart the kernel to use updated packages.


In [ ]:
import os
from langchain_chroma import Chroma
from langchain_ollama.embeddings import OllamaEmbeddings
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_ollama import OllamaLLM
from langchain.chains import RetrievalQA
from langchain_community.document_loaders import TextLoader
from langchain_community.document_loaders import DirectoryLoader

llm = OllamaLLM(model="llama3.2")


In [ ]:
loader = DirectoryLoader(file_path, glob="**/*.txt", loader_cls = TextLoader,loader_kwargs={"encoding": "utf-8"})
documents = loader.load()

In [ ]:
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
texts = text_splitter.split_documents(documents)

In [ ]:
len(texts)

233

In [ ]:
texts[4]

Document(metadata={'source': 'C:\\Users\\darkw\\PycharmProjects\\LetsBegin\\HTML\\05-03-ai-powered-supply-chain-startup-pando-lands-30m-investment.txt'}, page_content='Pando also taps algorithms and forms of machine learning to make predictions around supply chain events. For example, the platform attempts to match customer orders with suppliers, customers through the “right” channel (in terms of aspects like cost and carbon footprint) and fulfillment strategy (e.g. mode of freight, carrier, etc.). Beyond this, Pando can detect anomalies among deliveries, orders and freight invoices and anticipate supply chain risk given demand and supply trends.\n\nPando isn’t the only vendor doing this. Altana, which bagged $100 million in venture capital last October, uses an AI system to connect to and learn from logistics and business-to-business data — creating a shared view of supply chain networks. Everstream, another Pando rival, offers its own dashboards for data analysis, integrated with exi

In [ ]:
persist_directory = "db"

Create the DATABASE

In [ ]:
embedding = OllamaEmbeddings(model="nomic-embed-text")


In [ ]:
vectordb = Chroma.from_documents(documents=texts, embedding=embedding, persist_directory=persist_directory)

In [ ]:
print(dir(vectordb))

['_Chroma__ensure_collection', '_Chroma__query_collection', '_LANGCHAIN_DEFAULT_COLLECTION_NAME', '__abstractmethods__', '__annotations__', '__class__', '__delattr__', '__dict__', '__dir__', '__doc__', '__eq__', '__format__', '__ge__', '__getattribute__', '__getstate__', '__gt__', '__hash__', '__init__', '__init_subclass__', '__le__', '__lt__', '__module__', '__ne__', '__new__', '__reduce__', '__reduce_ex__', '__repr__', '__setattr__', '__sizeof__', '__slots__', '__str__', '__subclasshook__', '__weakref__', '_abc_impl', '_asimilarity_search_with_relevance_scores', '_chroma_collection', '_client', '_client_settings', '_collection', '_collection_metadata', '_collection_name', '_cosine_relevance_score_fn', '_embedding_function', '_euclidean_relevance_score_fn', '_get_retriever_tags', '_max_inner_product_relevance_score_fn', '_persist_directory', '_select_relevance_score_fn', '_similarity_search_with_relevance_scores', 'aadd_documents', 'aadd_texts', 'add_documents', 'add_images', 'add_tex

In [ ]:
vectordb.persist()
vectordb = None

AttributeError: 'Chroma' object has no attribute 'persist'

In [ ]:
vectordb = Chroma(persist_directory=persist_directory, embedding_function=embedding)

       Make a retriver
       

In [ ]:
retriever = vectordb.as_retriever()

In [ ]:
docs = retriever.invoke("How much money did Pando raise?")

In [ ]:
len(docs)

4

In [ ]:
retriever = vectordb.as_retriever(search_kwargs={"k":2})

In [ ]:
retriever.search_type

'similarity'

In [ ]:
retriever.search_kwargs

{'k': 2}

In [ ]:
llm = OllamaLLM(model="llama3.2")



In [ ]:
qa_chain = RetrievalQA.from_chain_type(llm=llm, chain_type="stuff", retriever = retriever, return_source_documents = True)

In [ ]:
def process_llm_response(llm_response):
    print(llm_response['result'])
    print('\n\nSources:')
    for source in llm_response["source_documents"]:
        print(source.metadata['source'])

In [ ]:
query = "How much money did Pando raise?"
llm_response = qa_chain.invoke({"query": query})
process_llm_response(llm_response)


Pando raised $45 million in total, including the $30 million in the recent Series B round.


Sources:
C:\Users\darkw\PycharmProjects\LetsBegin\HTML\05-03-ai-powered-supply-chain-startup-pando-lands-30m-investment.txt
C:\Users\darkw\PycharmProjects\LetsBegin\HTML\05-03-ai-powered-supply-chain-startup-pando-lands-30m-investment.txt


In [ ]:
query = input("Enter your query : ")
llm_response = qa_chain(query)
# process_llm_response(llm_response)
llm_response

Enter your query :  How much money did Pando raise?


{'query': 'How much money did Pando raise?',
 'result': 'Pando raised $45 million in total, with the latest Series B round bringing that amount up to $30 million, for a total of $45 million.',
 'source_documents': [Document(metadata={'source': 'C:\\Users\\darkw\\PycharmProjects\\LetsBegin\\HTML\\05-03-ai-powered-supply-chain-startup-pando-lands-30m-investment.txt'}, page_content='Signaling that investments in the supply chain sector remain robust, Pando, a startup developing fulfillment management technologies, today announced that it raised $30 million in a Series B round, bringing its total raised to $45 million.\n\nIron Pillar and Uncorrelated Ventures led the round, with participation from existing investors Nexus Venture Partners, Chiratae Ventures and Next47. CEO and founder Nitin Jayakrishnan says that the new capital will be put toward expanding Pando’s global sales, marketing and delivery capabilities.\n\n“We will not expand into new industries or adjacent product areas,” he t

In [ ]:
def format_llm_response(llm_response):
    """Format the LLM response for better readability."""

    # Extracting the relevant parts of the response
    query = llm_response.get('query', 'No query provided')
    result = llm_response.get('result', 'No result available')
    source_documents = llm_response.get('source_documents', [])

    # Start building the output
    output = []
    output.append(f"Query: {query}\n")

    # Format the result
    output.append(f"Result: {result}\n")

    # Format the source documents
    if source_documents:
        output.append("Source Documents:\n")
        for doc in source_documents:
            source = doc.metadata.get('source', 'Unknown source')
            content = doc.page_content
            output.append(f"  Source: {source}\n")
            output.append(f"  Content: {content}\n")
    else:
        output.append("No source documents available.\n")

    # Join the output list into a single string
    return ''.join(output)

# Example usage
while True:
    query = input("Enter your query: ")
    llm_response = qa_chain.invoke(query)  # Use invoke instead of call

    # Format and print the response
    formatted_output = format_llm_response(llm_response)
    print(formatted_output)

Enter your query:  AI


Query: AI
Result: I'd be happy to help! However, I notice that the provided text is a passage about AI-powered coding tools, but it doesn't ask a specific question. Could you please provide a question related to the topic of AI or code-generating systems? I'll do my best to answer it based on the context provided.
Source Documents:
  Source: C:\Users\darkw\PycharmProjects\LetsBegin\HTML\05-04-hugging-face-and-servicenow-release-a-free-code-generating-model.txt
  Content: AI startup Hugging Face and ServiceNow Research, ServiceNow’s R&D division, have released StarCoder, a free alternative to code-generating AI systems along the lines of GitHub’s Copilot.

Code-generating systems like DeepMind’s AlphaCode; Amazon’s CodeWhisperer; and OpenAI’s Codex, which powers Copilot, provide a tantalizing glimpse at what’s possible with AI within the realm of computer programming. Assuming the ethical, technical and legal issues are someday ironed out (and AI-powered coding tools don’t cause more bu

Enter your query:  can you use ai in various technologies


Query: can you use ai in various technologies
Result: Yes, AI can be used in a wide range of technologies beyond just code-generating systems. Some examples include:

* Virtual assistants (e.g. Siri, Alexa)
* Image and facial recognition
* Natural Language Processing (NLP) for language translation, sentiment analysis, and text summarization
* Predictive analytics and machine learning for decision-making in industries like healthcare, finance, and marketing
* Autonomous vehicles and robotics
* Cybersecurity systems to detect and prevent threats

This is not an exhaustive list, but it gives you an idea of the many technologies where AI can be applied.
Source Documents:
  Source: C:\Users\darkw\PycharmProjects\LetsBegin\HTML\05-04-hugging-face-and-servicenow-release-a-free-code-generating-model.txt
  Content: AI startup Hugging Face and ServiceNow Research, ServiceNow’s R&D division, have released StarCoder, a free alternative to code-generating AI systems along the lines of GitHub’s Copi

KeyboardInterrupt: Interrupted by user